In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

The types `literal`, `clause`, and `cnf` are defined in the notebook `04-CNF.ipynb`:
```
type literal =
  | Pos of string    (* p *)
  | Neg of string    (* ¬p *)

type clause = literal RSet.t
type cnf    = clause RSet.t
```

In [ ]:
import_notebook "04-CNF.ipynb"

# Refutational Completeness of the Cut Rule

This notebook implements a number of procedures that are needed in our proof of the <em style="color:blue">refutational completeness</em> of the cut rule.

The function `complement` takes one argument:
- `l` is a literal.

It computes the *complement* of `l`.
As we are only dealing with formulas in CNF that are written in set notation and these formulas do not contain $\top$ or $\bot$, a literal is either
a propositional variable or the negation of a propositional variable.
Therefore, if $p$ is a propositional variable, we have the following:
* $\texttt{complement}(p) = \neg p$,
* $\texttt{complement}(\neg p) = p$.

In [ ]:
let complement (l : literal) : literal =
  match l with
  | Pos p -> Neg p
  | Neg p -> Pos p

In [ ]:
complement (Pos "p")

In [ ]:
complement (Neg "p")

The function `extract_variable` takes one argument:
- `l` is a literal.

It returns the propositional variable of `l`.  If $p$ is a propositional variable, we have the following:

1. $\texttt{extract\_variable}(p) = p$,
2. $\texttt{extract\_variable}(\neg p) = p$.

The or-pattern `Pos p | Neg p` matches both kinds of literals.

In [ ]:
let extract_variable (l : literal) : string =
  match l with
  | Pos p | Neg p -> p

In [ ]:
extract_variable (Pos "p")

In [ ]:
extract_variable (Neg "p")

The function `collect_variables` takes one argument:
- `m` is a set of clauses.

It returns the set of all propositional variables occurring in `m`.  For a clause `c`, the expression `RSet.map extract_variable c` computes the set of the variables of `c`.  `RSet.flat_map` computes the union of these sets for all clauses of `m`.

In [ ]:
let collect_variables (m : cnf) : string RSet.t =
  RSet.flat_map (RSet.map extract_variable) m

In [ ]:
let c1 = RSet.of_list [Pos "p"; Pos "q"; Pos "r"]
let c2 = RSet.of_list [Neg "p"; Neg "q"; Neg "s"]
let m  = RSet.of_list [c1; c2];;
collect_variables m

The function `prettify_clause` takes one argument:
- `c` is a clause.

It converts `c` into a string, e.g. `"{¬p, q}"`.

In [ ]:
let prettify_clause (c : clause) : string =
  let literal_to_string = function
    | Pos p -> p
    | Neg p -> "¬" ^ p
  in
  RSet.to_string literal_to_string c

The function `prettify` takes one argument:
- `m` is a set of clauses.

It converts `m` into a string that looks like a set of sets.

In [ ]:
let prettify (m : cnf) : string =
  RSet.to_string prettify_clause m

The function `cut_rule` takes two arguments:
- `c1` is a clause,
- `c2` is a clause.

It returns the set of all clauses that can be derived from `c1` and `c2` using the *cut rule*.  In set notation, the cut rule is the following rule of inference:
$$
   \frac{\displaystyle \;C_1\cup \{l\} \quad C_2 \cup \bigl\{\overline{\,l\,}\bigr\}}{\displaystyle C_1 \cup C_2}
$$

**Implementation:** For every literal `l` of `c1` whose complement `l_bar` occurs in `c2`, the clause $(c_1 \backslash \{l\}) \cup (c_2 \backslash \{\overline{\,l\,}\})$ is derived.  `RSet.filter_map` collects these clauses: For a literal `l` whose complement does not occur in `c2`, the function returns `None`.

In [ ]:
let cut_rule (c1 : clause) (c2 : clause) : cnf =
  RSet.filter_map
    (fun l ->
       let l_bar = complement l in
       if RSet.mem l_bar c2 then
         Some (RSet.union (RSet.remove l c1) (RSet.remove l_bar c2))
       else
         None)
    c1

In [ ]:
let c1 = RSet.of_list [Neg "q"; Neg "r"]
let c2 = RSet.of_list [Pos "q"; Pos "r"];;
prettify (cut_rule c1 c2)

The function `saturate` takes one argument:
- `clauses` is a set of *clauses*, where each clause is a set of *literals*.

It computes the set of all clauses that can be derived from clauses in the set `clauses` using the *cut rule*.  The function keeps applying the cut rule until either no new clauses can be derived, or the empty clause $\{\}$ is derived.  The resulting set of clauses is *saturated* in the following sense:  If $C_1$ and $C_2$ are clauses from the resulting set and the clause $D$ can be derived from $C_1$ and $C_2$ via the cut rule, then $D$ is an element of the resulting set or the resulting set is $\bigl\{\{\}\bigr\}$.

**Implementation:** The recursive function `loop` takes the set `current` of clauses derived so far.
1. It computes the set `pairs` of all pairs of clauses from `current`.
2. `derived` is the set of all clauses that can be derived from these pairs via the cut rule.  The function `fun (c1, c2) -> cut_rule c1 c2` takes a pair and *destructures* it into its components `c1` and `c2`.
3. If the empty clause has been derived, the result is the set $\bigl\{\{\}\bigr\}$.
4. Otherwise, the derived clauses are added to `current`.  If this does not change the number of clauses, no new clause has been derived and `current` is saturated.  Otherwise, `loop` is called again with the bigger set.

In [ ]:
let saturate (clauses : cnf) : cnf =
  let rec loop (current : cnf) : cnf =
    let pairs   = RSet.cartesian_product current current in
    let derived = RSet.flat_map (fun (c1, c2) -> cut_rule c1 c2) pairs in
    if RSet.mem RSet.empty derived then
      RSet.singleton RSet.empty
    else
      let next = RSet.union current derived in
      if RSet.cardinal next = RSet.cardinal current then current else loop next
  in
  loop clauses

In [ ]:
let c1 = RSet.of_list [Pos "p"; Pos "q"]
let c2 = RSet.of_list [Neg "p"]
let c3 = RSet.of_list [Neg "p"; Neg "q"];;
prettify (saturate (RSet.of_list [c1; c2; c3]))

The function `find_valuation` takes one argument:
- `clauses` is a set of clauses.

The function tries to compute a variable interpretation that makes all of these clauses `true`.  The result has the type `literal RSet.t option`:
- If the attempt is successful, `Some literals` is returned.  The set `literals` does not contain any complementary literals and therefore corresponds to a variable assignment satisfying all clauses.
- If `clauses` is unsatisfiable, `None` is returned.

**Implementation:** First, `clauses` is saturated.  If the saturated set contains the empty clause, `clauses` is unsatisfiable.  Otherwise, the variables are processed one by one in ascending order.  `RSet.fold` passes the set `literals` of the literals chosen so far from one variable to the next.  A variable `p` has to be true if there is a clause in the saturated set that contains `p` and whose other literals are all false, i.e. whose other literals have complements that have already been chosen.  In this case, the literal `Pos p` is added to `literals`, otherwise the literal `Neg p` is added.

In [ ]:
let find_valuation (clauses : cnf) : literal RSet.t option =
  let saturated = saturate clauses in
  if RSet.mem RSet.empty saturated then
    None
  else
    let choose_literal (p : string) (literals : literal RSet.t) : literal RSet.t =
      (* A clause forces p to be true if it contains p and all other literals
         in the clause evaluate to false, i.e. their complements are already
         in the set literals. *)
      let forces_p =
        RSet.exists
          (fun c -> RSet.mem (Pos p) c
                    && RSet.for_all (fun l -> RSet.mem (complement l) literals)
                                    (RSet.remove (Pos p) c))
          saturated
      in
      RSet.add (if forces_p then Pos p else Neg p) literals
    in
    Some (RSet.fold choose_literal (collect_variables clauses) RSet.empty)

In [ ]:
let c01 = RSet.of_list [Pos "r"; Pos "p"; Pos "s"]
let c02 = RSet.of_list [Pos "r"; Pos "s"]
let c03 = RSet.of_list [Pos "p"; Pos "q"; Pos "s"]
let c04 = RSet.of_list [Neg "p"; Neg "q"]
let c05 = RSet.of_list [Neg "p"; Pos "s"; Neg "r"]
let c06 = RSet.of_list [Pos "p"; Neg "q"; Pos "r"]
let c07 = RSet.of_list [Neg "r"; Neg "s"; Pos "q"]
let c08 = RSet.of_list [Neg "p"; Neg "s"]
let c09 = RSet.of_list [Pos "p"; Neg "r"; Neg "q"]
let c10 = RSet.of_list [Neg "p"; Pos "r"; Pos "q"; Neg "s"]
let c11 = RSet.of_list [Pos "p"; Pos "q"; Pos "r"; Neg "s"]

let clauses1 = RSet.of_list [c01; c02; c03; c04; c05; c06; c07; c08; c09; c10];;

match find_valuation clauses1 with
| Some valuation -> print_endline (prettify_clause valuation)
| None           -> print_endline "false"

In [ ]:
let clauses2 = RSet.of_list [c01; c02; c03; c04; c05; c06; c07; c08; c09; c10; c11];;

match find_valuation clauses2 with
| Some valuation -> print_endline (prettify_clause valuation)
| None           -> print_endline "false"

In [ ]:
saturate clauses2